<a href="https://colab.research.google.com/github/ChenHY1217/Projects-In-MLAI-LABS/blob/main/LAB3/PIMLAI_Lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3

Environment Setup

In [ ]:
!pip install pillow

Starter Imports

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import time
from PIL import Image

# Week 1 - NLP Pipelines

## Part A - Sentiment Analysis and Topic Classification via Zero-Shot Classification

Sentiment Analysis

In [ ]:
sentiment_pipe = pipeline("text-classification", truncation= True)

texts = [
    "I love this movie! It was my favorite!",
    "The weather today is terrible.",
    "This was the worst experience I've ever had.",
    "It wasn't my favorite, but not terrible either."
]

for t in texts:
    print(sentiment_pipe(t))

Topic classification via zero-shot classification

In [ ]:
classifier = pipeline("zero-shot-classification", truncation = True)

text = "The stock market crashed due to rising inflation and interest rates."

candidate_labels = ["finance", "sports", "politics", "technology"]

result = classifier(text, candidate_labels)

print(result)

## Part B - Dataset subset and evaluation

We will evaluate sentiment analysis.

In [ ]:
!pip install evaluate

In [ ]:
import evaluate
from datasets import load_dataset
dataset = load_dataset("imdb", split="test[:100]")

# Initialize accuracy metric
accuracy_metric = evaluate.load("accuracy")

predictions = []
true_labels = []
inference_times = []

# Map pipeline output to numerical labels (0 for negative, 1 for positive)
def map_label_to_int(label_str):
    return 1 if label_str == 'POSITIVE' else 0

print(f"Evaluating on {len(dataset)} examples...")

for example in dataset:
    start_time = time.time()
    # The sentiment_pipe returns a list of dictionaries, take the first element
    result = sentiment_pipe(example['text'])[0]
    end_time = time.time()

    inference_times.append(end_time - start_time)
    predictions.append(map_label_to_int(result['label']))
    true_labels.append(example['label'])

# Compute metrics
accuracy = accuracy_metric.compute(predictions=predictions, references=true_labels)
avg_inference_time = sum(inference_times) / len(inference_times)

print(f"\nEvaluation Results:")
print(f"Accuracy: {accuracy['accuracy']:.4f}")
print(f"Average Inference Time per Example: {avg_inference_time:.4f} seconds")

##Part C Model comparison

In [ ]:
distil = pipeline('text-classification', model="distilbert-base-uncased-finetuned-sst-2-english", truncation=True)
cardiffnlp = pipeline('text-classification', model="cardiffnlp/twitter-roberta-base-sentiment", truncation=True)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import time
from transformers import pipeline # Added import for pipeline


def map_label_to_int(label, model_name):
    if model_name == "distil":
        return 1 if label == "POSITIVE" else 0

    elif model_name == "Cardiffnlp":
        val = int(label.split("_")[1])  # 0–4
        return 1 if val >= 2 else 0  # 2,3,4 → positive


results = []

distil = pipeline('text-classification', model="distilbert-base-uncased-finetuned-sst-2-english", truncation=True)

cardiffnlp = pipeline('text-classification', model="cardiffnlp/twitter-roberta-base-sentiment", truncation=True, max_length=128)

for model_name, model in [("distil", distil), ("Cardiffnlp", cardiffnlp)]:
    predictions = []
    true_labels = []
    inference_times = []

    for example in dataset:
        start_time = time.time()
        # truncation=True is now set during pipeline initialization, no need to pass it here.
        result = model(example['text'])[0]
        end_time = time.time()

        inference_times.append(end_time - start_time)
        predictions.append(map_label_to_int(result['label'], model_name))

        # true labels per model
        if model_name == "distil":
            true_labels.append(example['label'])  # 0=neg, 1=pos
        elif model_name == "Cardiffnlp":
            true_labels.append(example['label'])  # keep 0/1

    accuracy = accuracy_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions, average="weighted")
    avg_time_ms = sum(inference_times)/len(inference_times)*1000

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "F1 Score": f1,
        "Avg Inference Time (ms)": avg_time_ms
    })

import pandas as pd
smallComparTable = pd.DataFrame(results)
print(smallComparTable)

Distilbert
---


DistilBERT is fine-tuned on the SST-2 dataset (short English movie reviews), making it well-suited for IMDB sentiment classification. It is fast and efficient, but limited to 512 tokens and can struggle with long or nuanced text (e.g., sarcasm). Its intended use is to be used as a base which is fine tuned by the user. Out of scope problems would be using it to train in text generation as an metric of accurate real world events events. One of the models limitations is on smaller populations producing biased information. License: Apache-2.0


Cardiffnlp
---

Cardiffnlp is ment to handle social media posts and catorgorizinging them. An example of an out of scope problem would be using it for say reviews or resturants or movies. Limitations it is only able to handle shorter texts no like 1000 word essays. Lincense MITDistilbert



### Surprising Behavior

One surprising behavior was that although CardiffNLP Twitter RoBERTa Sentiment Model is designed for short social media posts, it was applied to IMDB movie reviews, which we thought would generalize well with shorter social media posts. Surprisingly, it performed slightly worse than DistilBERT SST-2 (0.88 vs 0.89 accuracy, 0.936 vs 0.942 F1), even though it is a larger, more versatile model. This behavior is likely due to a combination of domain mismatch (social media posts vs. movie reviews) and input length limitations, which prevent CardiffNLP from effectively handling longer reviews.

One concrete failure case for DistilBERT in particular would be the use of nuanced text like sarcasm (as mentioned before). The model would be unable to detect the difference between someone making a sarcastic comment about something and a genuine comment, leading to classification failures.

# Week 2

## Environment Setup

In [ ]:
pip install transformers torch pillow

## Part A - Implementation of Image Classification and Image Captioning

### Image Classification

In [ ]:
from transformers import pipeline
from PIL import Image
import requests
import warnings
warnings.filterwarnings('ignore')

# Load pipeline
classifier = pipeline("image-classification")

# Load image
url = "https://www.cats.org.uk/media/13139/220325case013.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# Run model
results = classifier(image)

### Display Results

In [ ]:
display(image)

# Print results
for r in results:
    print(f"{r['label']}: {round(r['score'], 4)}")

### Image Captioning

In [ ]:
from transformers import pipeline
from PIL import Image
import requests

# Load pipeline
captioner = pipeline("image-text-to-text")

# Load image
url = "https://www.cats.org.uk/media/13139/220325case013.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# Generate caption
result = captioner(image)

### Display Results

In [ ]:
display(image)

print(result[0]['generated_text'])

## Part B

In [ ]:
import kagglehub
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

src_path = kagglehub.dataset_download("zalando-research/fashionmnist")
dst_path = "/content/fashionmnist"

shutil.copytree(src_path, dst_path, dirs_exist_ok=True)

print("Saved to:", dst_path)
df = pd.read_csv("/content/fashionmnist/fashion-mnist_train.csv")
import pandas as pd
import numpy as np
from PIL import Image
import os

df = pd.read_csv("/content/fashionmnist/fashion-mnist_train.csv")

os.makedirs("digit_images2", exist_ok=True)
df = df[:100]

In [ ]:
images = []
labels = []
labelIndex = ["T-shirt", "Pants", "Long Sleeve Shirt", "Dress", "Buttoned Shirt", "Sandles", "Coat", "Sneakers", "Purse", "Boots"]
for idx, row in df.iterrows():
    label = row['label']
    label = labelIndex[label]
    pixels = row[1:].values
    pixels = pixels.reshape(28,28).astype(np.uint8)
    img = Image.fromarray(pixels, mode='L')
    img.save(f"digit_images2/{idx}.png")
    if idx < 5:
        img.show()
    labels.append(label)
    images.append(f"digit_images2/{idx}.png")



In [ ]:
from PIL import Image
import os

image_folder = "digit_images2"
image_files = os.listdir(image_folder)

for filename in image_files:
    path = os.path.join(image_folder, filename)
    img = Image.open(path).convert("RGB")
    print(f"Loaded {filename}, size={img.size}")

In [ ]:
from transformers import pipeline
from PIL import Image
import warnings
import os
warnings.filterwarnings('ignore')


classifier = pipeline("image-classification", model="google/vit-base-patch16-224", device=-1)
detector = pipeline("object-detection", model="facebook/detr-resnet-50", device=-1)
segmenter = pipeline("image-segmentation", model="nvidia/segformer-b0-finetuned-ade-512-512", device=-1)
captioner = pipeline(task="image-text-to-text", model="Salesforce/blip-image-captioning-base", device=-1)

In [ ]:
for i, filename in enumerate(image_files[:5]):
    path = os.path.join(image_folder, filename)
    img = Image.open(path).convert("RGB").resize((224, 224))
    img_up = img.resize((224, 224))

    class_result = classifier(img_up)
    detect_result = detector(img_up)
    segment_result = segmenter(img_up)
    caption_result = captioner(img_up, "")

    plt.imshow(img_up)
    plt.axis('off')
    plt.title(f"{filename} - {caption_result[0]['generated_text']}")
    plt.show()

    print(f"File: {filename}")
    print("Classification:", class_result)
    print("Detection:", detect_result)
    print("Segmentation:", segment_result)
    print("Caption:", caption_result[0]['generated_text'])
    print("-"*200)

### Part C - Model Comparison

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
import os
import time
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore') # Suppress warnings from Hugging Face pipelines

# Load the CIFAR-100 dataset for demonstration
print("Loading CIFAR-100 dataset...")
dataset = load_dataset("cifar100", split="test[:50]") # Using a small test split for speed
print(f"Loaded {len(dataset)} examples from CIFAR-100.")

# Initialize models
print("Loading models...")
classifier_vit = pipeline("image-classification", model="google/vit-base-patch16-224")
classifier_resnet = pipeline("image-classification", model="microsoft/resnet-50")
print("Models loaded.")

# CIFAR-100's fine_label_names
cifar100_label_names = dataset.features['fine_label'].names
cifar100_label2id = {name: i for i, name in enumerate(cifar100_label_names)}

# Enhanced Helper function to map ImageNet-like labels to CIFAR-100 categories (heuristic)
def map_imagenet_to_cifar100(imagenet_label):
    label = imagenet_label.lower()

    # Category: 'airplane' (0)
    if any(x in label for x in ["airplane", "airliner", "jet", "plane", "aircraft", "cessna"]):
        return cifar100_label2id.get("airplane", -1)

    # Category: 'automobile' (1) and 'pickup_truck' (58), 'truck' (90), 'tank' (86), 'tractor' (89)
    if any(x in label for x in ["car", "automobile", "sedan", "coupe", "convertible", "limousine", "sports car", "racing car"]):
        return cifar100_label2id.get("automobile", -1)
    if any(x in label for x in ["pickup", "pickup truck", "fire engine", "ambulance", "recreational vehicle", "tow truck", "garbage truck"]):
        return cifar100_label2id.get("pickup_truck", -1) # More specific for truck types
    if "truck" in label:
        return cifar100_label2id.get("truck", -1)
    if "tank" in label:
        return cifar100_label2id.get("tank", -1)
    if "tractor" in label:
        return cifar100_label2id.get("tractor", -1)

    # Category: 'bird' (multiple types)
    if any(x in label for x in ["bird", "finch", "sparrow", "robin", "owl", "eagle", "parrot", "canary", "hummingbird", "vulture", "goose", "duck", "chicken"]):
        return cifar100_label2id.get("bird", -1) # Generic bird mapping
    if "eagle" in label:
        return cifar100_label2id.get("eagle", -1) # Example for specific bird if CIFAR-100 had it

    # Category: 'cat' (17), 'tiger' (88), 'lion' (43), 'leopard' (42)
    if any(x in label for x in ["cat", "tabby", "siamese", "persian cat", "egyptian cat", "lynx"]):
        return cifar100_label2id.get("cat", -1)
    if "tiger" in label:
        return cifar100_label2id.get("tiger", -1)
    if "lion" in label:
        return cifar100_label2id.get("lion", -1)
    if "leopard" in label:
        return cifar100_label2id.get("leopard", -1)

    # Category: 'deer' (29), 'horse' (39), 'cattle' (19), 'camel' (15), 'elephant' (31), 'bear' (3), 'fox' (34), 'kangaroo' (38), 'mouse' (49), 'otter' (55), 'rabbit' (63), 'raccoon' (64), 'squirrel' (80), 'wolf' (96)
    if any(x in label for x in ["deer", "stag", "fawn"]):
        return cifar100_label2id.get("deer", -1)
    if any(x in label for x in ["dog", "poodle", "husky", "labrador", "dalmatian", "german shepherd", "golden retriever", "siberian husky"]):
        return cifar100_label2id.get("dog", -1)
    if any(x in label for x in ["frog", "tree frog", "bullfrog"]):
        return cifar100_label2id.get("frog", -1)
    if any(x in label for x in ["horse", "pony"]):
        return cifar100_label2id.get("horse", -1)
    if any(x in label for x in ["ship", "boat", "liner", "freighter", "vessel", "yacht", "sailboat", "gondola"]):
        return cifar100_label2id.get("ship", -1)
    if any(x in label for x in ["cattle", "cow", "ox"]):
        return cifar100_label2id.get("cattle", -1)
    if "camel" in label:
        return cifar100_label2id.get("camel", -1)
    if "elephant" in label:
        return cifar100_label2id.get("elephant", -1)
    if "bear" in label:
        return cifar100_label2id.get("bear", -1)
    if "fox" in label:
        return cifar100_label2id.get("fox", -1)
    if "kangaroo" in label:
        return cifar100_label2id.get("kangaroo", -1)
    if "mouse" in label:
        return cifar100_label2id.get("mouse", -1)
    if "otter" in label:
        return cifar100_label2id.get("otter", -1)
    if "rabbit" in label:
        return cifar100_label2id.get("rabbit", -1)
    if "raccoon" in label:
        return cifar100_label2id.get("raccoon", -1)
    if "squirrel" in label:
        return cifar100_label2id.get("squirrel", -1)
    if "wolf" in label:
        return cifar100_label2id.get("wolf", -1)
    if "snake" in label:
        return cifar100_label2id.get("snake", -1)
    if "lizard" in label:
        return cifar100_label2id.get("lizard", -1)
    if "spider" in label:
        return cifar100_label2id.get("spider", -1)
    if "turtle" in label:
        return cifar100_label2id.get("turtle", -1)
    if "whale" in label:
        return cifar100_label2id.get("whale", -1)
    if "fish" in label:
        return cifar100_label2id.get("aquarium_fish", -1) # Generic fish mapping
    if "butterfly" in label:
        return cifar100_label2id.get("butterfly", -1)
    if "beetle" in label:
        return cifar100_label2id.get("beetle", -1)
    if "bee" in label:
        return cifar100_label2id.get("bee", -1)
    if "snail" in label:
        return cifar100_label2id.get("snail", -1)
    if "worm" in label:
        return cifar100_label2id.get("worm", -1)
    if "crab" in label:
        return cifar100_label2id.get("crab", -1)
    if "lobster" in label:
        return cifar100_label2id.get("lobster", -1)
    if "shark" in label:
        return cifar100_label2id.get("shark", -1)
    if "dolphin" in label:
        return cifar100_label2id.get("dolphin", -1)
    if "seal" in label:
        return cifar100_label2id.get("seal", -1)
    if "crocodil" in label:
        return cifar100_label2id.get("crocodile", -1)

    # Objects and other
    if "bicycle" in label or "bike" in label:
        return cifar100_label2id.get("bicycle", -1)
    if "bottle" in label:
        return cifar100_label2id.get("bottle", -1)
    if "bowl" in label:
        return cifar100_label2id.get("bowl", -1)
    if "bridge" in label:
        return cifar100_label2id.get("bridge", -1)
    if "bus" in label:
        return cifar100_label2id.get("bus", -1)
    if "castle" in label:
        return cifar100_label2id.get("castle", -1)
    if "chair" in label:
        return cifar100_label2id.get("chair", -1)
    if "clock" in label:
        return cifar100_label2id.get("clock", -1)
    if "cloud" in label:
        return cifar100_label2id.get("cloud", -1)
    if "couch" in label or "sofa" in label:
        return cifar100_label2id.get("couch", -1)
    if "cup" in label or "coffee mug" in label:
        return cifar100_label2id.get("cup", -1)
    if "house" in label or "home" in label:
        return cifar100_label2id.get("house", -1)
    if "keyboard" in label:
        return cifar100_label2id.get("keyboard", -1)
    if "lamp" in label:
        return cifar100_label2id.get("lamp", -1)
    if "motorcycle" in label or "motorbike" in label:
        return cifar100_label2id.get("motorcycle", -1)
    if "plate" in label:
        return cifar100_label2id.get("plate", -1)
    if "rocket" in label:
        return cifar100_label2id.get("rocket", -1)
    if "table" in label:
        return cifar100_label2id.get("table", -1)
    if "telephone" in label or "phone" in label:
        return cifar100_label2id.get("telephone", -1)
    if "television" in label or "monitor" in label:
        return cifar100_label2id.get("television", -1)
    if "train" in label:
        return cifar100_label2id.get("train", -1)
    if "wardrobe" in label:
        return cifar100_label2id.get("wardrobe", -1)
    if "bed" in label:
        return cifar100_label2id.get("bed", -1)
    if "skyscraper" in label:
        return cifar100_label2id.get("skyscraper", -1)

    # Trees and Plants
    if "tree" in label:
        # Check for specific tree types if present in CIFAR-100
        if "maple" in label: return cifar100_label2id.get("maple_tree", -1)
        if "oak" in label: return cifar100_label2id.get("oak_tree", -1)
        if "pine" in label: return cifar100_label2id.get("pine_tree", -1)
        if "palm" in label: return cifar100_label2id.get("palm_tree", -1)
        if "willow" in label: return cifar100_label2id.get("willow_tree", -1)
        return cifar100_label2id.get("tree", -1) # Generic tree
    if "forest" in label:
        return cifar100_label2id.get("forest", -1)
    if "flower" in label or "rose" in label or "poppy" in label or "tulip" in label or "orchid" in label or "sunflower" in label:
        # More specific flower mappings if needed
        if "rose" in label: return cifar100_label2id.get("rose", -1)
        if "poppy" in label: return cifar100_label2id.get("poppy", -1)
        if "tulip" in label: return cifar100_label2id.get("tulip", -1)
        if "orchid" in label: return cifar100_label2id.get("orchid", -1)
        if "sunflower" in label: return cifar100_label2id.get("sunflower", -1)
        return cifar100_label2id.get("flower", -1) # Generic flower

    # Fruits and Vegetables
    if "apple" in label:
        return cifar100_label2id.get("apple", -1)
    if "orange" in label:
        return cifar100_label2id.get("orange", -1)
    if "pear" in label:
        return cifar100_label2id.get("pear", -1)
    if "pepper" in label:
        return cifar100_label2id.get("sweet_pepper", -1)

    # People categories
    if "boy" in label:
        return cifar100_label2id.get("boy", -1)
    if "girl" in label:
        return cifar100_label2id.get("girl", -1)
    if "man" in label:
        return cifar100_label2id.get("man", -1)
    if "woman" in label:
        return cifar100_label2id.get("woman", -1)
    if "baby" in label:
        return cifar100_label2id.get("baby", -1)

    # Landscapes / natural scenes
    if "mountain" in label:
        return cifar100_label2id.get("mountain", -1)
    if "sea" in label or "ocean" in label:
        return cifar100_label2id.get("sea", -1)
    if "road" in label:
        return cifar100_label2id.get("road", -1)
    if "plain" in label:
        return cifar100_label2id.get("plain", -1)


    # If no direct or heuristic match, return -1
    return -1

models = {
    "ViT-Base": classifier_vit,
    "ResNet-50": classifier_resnet
}

N_SAMPLES = len(dataset) # Use the full loaded subset
predictions_dict = {name: [] for name in models}
true_labels_list = []
inference_times_dict = {name: [] for name in models}
raw_predictions_dict = {name: [] for name in models} # To store raw model outputs for analysis

print(f"Evaluating on {N_SAMPLES} CIFAR-100 examples...")

for i in range(N_SAMPLES):
    example = dataset[i]
    img = example['img'].convert("RGB").resize((224, 224)) # CIFAR-100 images are 32x32, resize for models
    true_label_id = example['fine_label'] # Use fine_label for CIFAR-100

    true_labels_list.append(true_label_id)

    for model_name, classifier_pipe in models.items():
        start_time = time.time()
        results = classifier_pipe(img)
        end_time = time.time()

        inference_times_dict[model_name].append(end_time - start_time)
        predicted_label_text = results[0]['label'] # Top ImageNet prediction string
        raw_predictions_dict[model_name].append(results) # Store full results

        # Map ImageNet prediction to CIFAR-100 label
        mapped_prediction_id = map_imagenet_to_cifar100(predicted_label_text)
        predictions_dict[model_name].append(mapped_prediction_id)


# Calculate metrics
results_summary = []
for model_name, predictions_ids in predictions_dict.items():
    # Filter out -1 predictions (unrecognized labels by our mapping)
    filtered_true_labels = []
    filtered_predictions = []
    for i in range(N_SAMPLES):
        if predictions_ids[i] != -1:
            filtered_predictions.append(predictions_ids[i])
            filtered_true_labels.append(true_labels_list[i])

    if len(filtered_predictions) > 0:
        accuracy = accuracy_score(filtered_true_labels, filtered_predictions)
        f1 = f1_score(filtered_true_labels, filtered_predictions, average="weighted", zero_division=0)
    else:
        accuracy = 0.0
        f1 = 0.0

    avg_inference_time_ms = (sum(inference_times_dict[model_name]) / len(inference_times_dict[model_name])) * 1000

    results_summary.append({
        "Model": model_name,
        "Accuracy (mapped)": f"{accuracy:.4f} ({len(filtered_predictions)}/{N_SAMPLES})",
        "F1 Score (mapped)": f"{f1:.4f}",
        "Avg Inference Time (ms)": f"{avg_inference_time_ms:.2f}"
    })

comparison_df = pd.DataFrame(results_summary)
print("\n--- Model Comparison Results on CIFAR-100 Subset (with heuristic mapping) ---")
print(comparison_df.to_markdown(index=False))

# --- Example where models differ meaningfully or show mapping issues ---
example_output_str = ""
found_illustrative_example = False
for i in range(N_SAMPLES):
    vit_pred_mapped = predictions_dict["ViT-Base"][i]
    resnet_pred_mapped = predictions_dict["ResNet-50"][i]
    true_label_id = true_labels_list[i]

    # Find an example where they predict differently AND at least one of them made a mapped prediction
    if vit_pred_mapped != resnet_pred_mapped and (vit_pred_mapped != -1 or resnet_pred_mapped != -1):
        ex_idx = i
        ex_true_label = cifar100_label_names[true_label_id]
        ex_vit_pred_name = cifar100_label_names[vit_pred_mapped] if vit_pred_mapped != -1 else 'Unmapped'
        ex_resnet_pred_name = cifar100_label_names[resnet_pred_mapped] if resnet_pred_mapped != -1 else 'Unmapped'

        example_output_str += f"\nFor an example image (index {ex_idx}) with true label '{ex_true_label}':\n"
        example_output_str += f"- ViT-Base mapped prediction: '{ex_vit_pred_name}'\n"
        example_output_str += f"- ResNet-50 mapped prediction: '{ex_resnet_pred_name}'\n"

        example_output_str += "\nRaw Top-3 ImageNet predictions:\n"
        example_output_str += "  ViT-Base:\n"
        for res in raw_predictions_dict["ViT-Base"][ex_idx][:3]:
            example_output_str += f"    - {res['label']} (score: {res['score']:.4f})\n"
        example_output_str += "  ResNet-50:\n"
        for res in raw_predictions_dict["ResNet-50"][ex_idx][:3]:
            example_output_str += f"    - {res['label']} (score: {res['score']:.4f})\n"

        example_output_str += "\nThis example demonstrates how the ImageNet-trained models interpret a CIFAR-100 image. While not directly trained on CIFAR-100, their general object recognition can sometimes align with a CIFAR-100 category through heuristic mapping, but also highlights discrepancies and the challenge of cross-dataset generalization without fine-tuning."
        found_illustrative_example = True
        break # Found a good example, stop searching

if not found_illustrative_example:
    example_output_str = "\nNo sufficiently differing example found within the sampled subset that resulted in different mapped predictions. This might happen if the models perform very similarly or the heuristic mapping is too coarse to capture subtle differences in the mapped output."

print(example_output_str)

ViT-Base (google/vit-base-patch16-224)

This a transformer built for vision purposes. It processes images as sequences of fixed-size patches similar to NLP and uses self-attention to capture global relationships within the image. It's pretrained on ImageNet-1k. This model is known for its ability to capture long-range dependencies. However, it can be computationally intense and thus requires larger datasets and compute time. For this task, it would perform slightly worse than ResNet-50. In addition, since this model is powerfully generic, it may not directly align with the broader FashionMNIST categories, introducing ambiguities.

ResNet-50 (microsoft/resnet-50)

This is a residual network. It is a classic CNN that utilizes skip connections to allow for much deeper networks, helping with mitigating the vanishing gradient problem and improving performance. It is also pre-trained on ImageNet-1k. These models are highly effective at hierarchical feature extraction. They provide a good balance between performance and computational efficiency. Therefore, it should show slightly better accuracy and faster compute time than ViT-Base.

Since this model is pre-trained, it may also have ambiguity due to the different output labels.